<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">1. Introduction to Data Interoperability</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 1.2 Lecture: Introduction to Open Table Formats

This lesson takes a closer look at the open table formats that sit underneath the modern lakehouse, comparing the anatomy and capabilities of Delta Lake and Apache Iceberg, explaining how UniForm bridges the two, and noting where Iceberg v3 fits in the picture today.

## Learning Objectives

By the end of this lesson, you will be able to:
- Trace the evolution from raw files to columnar storage to open table formats
- Describe the anatomy of a Unity Catalog managed table and an Iceberg table
- Compare the capabilities and optimizations of Delta and Iceberg
- Explain how UniForm exposes UC managed tables to Iceberg readers
- Place Iceberg v3 in the context of the format's evolution

## A. Evolution of Table Formats

Open table formats emerged from a long progression of file-based storage that gradually added the metadata and transactional guarantees needed for analytical workloads.

<div style="font-size: 1em; display: flex; align-items: center; justify-content: center; gap: 6px; flex-wrap: wrap; overflow-x: auto; max-width: 100%; margin: 16px 0;">
  <div style="background: #fff3e0; border: 2px solid #ff9800; padding: 12px 18px; border-radius: 6px; text-align: center;">
    <strong>Raw Files</strong><br/><span style="font-size: 0.85em; color: #555;">No guarantees<br/>Schema-on-read</span>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="background: #e3f2fd; border: 2px solid #1976d2; padding: 12px 18px; border-radius: 6px; text-align: center;">
    <strong>Columnar</strong><br/><span style="font-size: 0.85em; color: #555;">Parquet / ORC<br/>Better performance</span>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="display: flex; flex-direction: column; gap: 10px;">
    <div style="background: #0277bd; border: 2px solid #01579b; padding: 12px 18px; border-radius: 6px; text-align: center; color: #fff;">
      <img src="https://delta.io/_astro/delta-lake-logo.Dx7tzbyv_1OsUys.svg" height="24" style="margin-bottom: 4px;"/><br/>ACID transactions<br/>Time travel
    </div>
    <div style="background: #e0f2f1; border: 2px solid #009688; padding: 12px 18px; border-radius: 6px; text-align: center;">
      <span style="display: inline-block; background: #fff; padding: 4px 8px; border-radius: 4px;"><img src="https://iceberg.apache.org/assets/images/iceberg-logo-icon.png" height="30"/></span><br/><strong>Apache Iceberg</strong><br/><span style="font-size: 0.85em; color: #555;">ACID transactions<br/>Cross-engine</span>
    </div>
  </div>
  <div style="color: #999; font-size: 1.3em;">&#10132;</div>
  <div style="background: #f3e5f5; border: 2px solid #9c27b0; padding: 12px 18px; border-radius: 6px; text-align: center;">
    <strong>Unified Governance</strong><br/><span style="font-size: 0.85em; color: #555;">Unity Catalog</span>
  </div>
</div>

## B. Delta and Iceberg: The Core Table Formats

Both formats extend columnar storage (Parquet or ORC) with transaction logs for ACID guarantees:
- File-based with metadata management layers
- Support schema evolution, time travel, and data versioning
- Enable consistent reads during concurrent writes
- Optimized for both batch and streaming workloads

## C. Anatomy of a UC managed Table

<p style="font-size: 1em; line-height: 1.6; color: #333">A UC managed table is a directory of Parquet data files alongside a transaction log that records every change made to the table.</p>

<div style="padding: 18px 24px; background: #fafafa; border: 1px solid #e0e0e0; border-radius: 6px; max-width: 900px; margin: 16px 0; overflow-x: auto">
<table style="border-collapse: collapse; border: none; font-size: 1em; line-height: 1.65">
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555"><span style="color:#1B5162;font-weight:600">my_delta_table/</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">root table directory</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">├── <span style="color:#388e3c;font-weight:600">_delta_log/</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">transaction log, the source of truth</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── 00000000000000000000.json</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">commit at version 0 (<code>CREATE TABLE</code>)</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── 00000000000000000001.json</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">commit at version 1 (e.g. <code>INSERT</code>)</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── 00000000000000000002.json</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">commit at version 2 (e.g. <code>UPDATE</code>)</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   └── ...</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">other files</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">├── <span style="color:#c2410c">part-00000-abc-xxx.snappy.parquet</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">immutable Parquet data file</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">├── <span style="color:#c2410c">part-00001-def-xxx.snappy.parquet</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">immutable Parquet data file</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">└── <span style="color:#c2410c">part-00002-ghi-xxx.snappy.parquet</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">immutable Parquet data file</td></tr>
</table>
</div>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="visibility: hidden">&#x25B6;</span>
      <div>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li>The <strong>transaction log</strong> (<code>_delta_log</code>) is a directory of JSON commit files that record every change to the table; readers replay these to reconstruct the current snapshot.</li>
          <li><strong>Parquet data files</strong> hold the actual rows; the log only references them, so data files are immutable once written.</li>
          <li>Each commit creates an <strong>atomic version</strong> with snapshot isolation, which is what makes time travel and concurrent writes safe.</li>
          <li><strong>Statistics</strong> are collected per file at write time and stored in the log, enabling data skipping and query optimization without scanning data.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

## D. Anatomy of an Iceberg Table

<p style="font-size: 1em; line-height: 1.6; color: #333">An Iceberg table uses a layered metadata structure that separates table state, manifest lists, and manifest files to support cross-engine reads.</p>

<div style="padding: 18px 24px; background: #fafafa; border: 1px solid #e0e0e0; border-radius: 6px; max-width: 900px; margin: 16px 0; overflow-x: auto">
<table style="border-collapse: collapse; border: none; font-size: 1em; line-height: 1.65">
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555"><span style="color:#1B5162;font-weight:600">my_iceberg_table/</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">root table directory</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">├── <span style="color:#00796b;font-weight:600">metadata/</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">metadata layer, the source of truth</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── version-hint.text</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">pointer to current metadata version</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── v1.metadata.json</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">table state at version 1 (schema, partition spec, snapshots)</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── v2.metadata.json</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">table state at version 2</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── snap-xxx.avro</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">manifest list, one per snapshot</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   ├── xxx-m0.avro</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">manifest file, tracks data files + stats</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">│   └── ...</td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">other files</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">└── <span style="color:#00796b;font-weight:600">data/</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">data layer</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">    ├── <span style="color:#c2410c">data-00001.parquet</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">immutable Parquet data file</td></tr>
<tr><td style="border: none; padding: 0 24px 0 0; white-space: pre; font-family: ui-monospace, 'SF Mono', Menlo, Monaco, 'Courier New', monospace; font-size: 1em; color: #555">    └── <span style="color:#c2410c">data-00002.parquet</span></td><td style="border: none; padding: 0; font-size: 1em; color: #888; font-style: italic">immutable Parquet data file</td></tr>
</table>
</div>

<details>
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Expand for More Details</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="visibility: hidden">&#x25B6;</span>
      <div>
        <ul style="line-height: 1.8; color: #333; margin: 6px 0 14px 0">
          <li><strong>Metadata files</strong> capture table state at a point in time, including schema, partition spec, and snapshot history.</li>
          <li><strong>Manifest lists</strong> enumerate every manifest file that makes up a snapshot, which is how Iceberg supports cheap snapshot operations.</li>
          <li><strong>Manifest files</strong> track data files together with per-file statistics that drive partition and predicate pruning.</li>
          <li><strong><code>version-hint.text</code></strong> points to the latest metadata version, so readers can find the current snapshot without listing the metadata directory.</li>
        </ul>
      </div>
    </div>
  </div>
</details>

## E. Shared Capabilities of Delta and Iceberg

<p style="font-size: 1em; line-height: 1.6; color: #333">Despite their different internals, both formats expose the same set of capabilities to engines and users.</p>

<div style="font-size: 1em; display: grid; grid-template-columns: repeat(3, 1fr); gap: 16px; margin: 16px 0">

  <div style="border: 2px solid #1565C0; border-radius: 8px; overflow: hidden">
    <div style="background: #1565C0; color: #fff; padding: 10px 14px; font-weight: bold">Schema Evolution</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Add, rename, and reorder columns without rewriting data.</p>
    </div>
  </div>

  <div style="border: 2px solid #2E7D32; border-radius: 8px; overflow: hidden">
    <div style="background: #2E7D32; color: #fff; padding: 10px 14px; font-weight: bold">Time Travel</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Query historical versions of data with snapshot isolation.</p>
    </div>
  </div>

  <div style="border: 2px solid #E65100; border-radius: 8px; overflow: hidden">
    <div style="background: #E65100; color: #fff; padding: 10px 14px; font-weight: bold">ACID Transactions</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Atomic, consistent, isolated, and durable operations.</p>
    </div>
  </div>

  <div style="border: 2px solid #6A1B9A; border-radius: 8px; overflow: hidden">
    <div style="background: #6A1B9A; color: #fff; padding: 10px 14px; font-weight: bold">Complex Data Types</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Nested structures, arrays, and maps.</p>
    </div>
  </div>

  <div style="border: 2px solid #00838F; border-radius: 8px; overflow: hidden">
    <div style="background: #00838F; color: #fff; padding: 10px 14px; font-weight: bold">Metadata Optimizations</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Statistics-driven query planning and data skipping.</p>
    </div>
  </div>

  <div style="border: 2px solid #C62828; border-radius: 8px; overflow: hidden">
    <div style="background: #C62828; color: #fff; padding: 10px 14px; font-weight: bold">Streaming and Batch</div>
    <div style="padding: 16px; background: #fff">
      <p style="margin: 0; color: #333">Optimized for both real-time and batch workloads.</p>
    </div>
  </div>

</div>

## F. Performance Optimizations Comparison

The two formats use different mechanisms but converge on the same set of optimization techniques.

| Technique | Delta Lake | Apache Iceberg |
|-----------|-----------|----------------|
| **Data Organization** | Z-Ordering, Liquid Clustering | Position-based ordering |
| **Small File Handling** | `OPTIMIZE` command | `rewrite_data_files` procedure |
| **Statistics** | Min/max values, null count, per-column stats | Min/max values, null count, per-column stats |
| **Data Skipping** | Automatic based on statistics | Automatic based on statistics |
| **Cleanup Operations** | `VACUUM` command | `expire_snapshots` procedure |
| **Advanced Features** | Automatic Liquid Clustering, Deletion Vectors | Positional delete files, Deletion Vectors (v3) |
| **Partitioning** | Dynamic with auto-optimization | Hidden partitioning with evolution |
| **Caching** | Delta Cache for metadata | Metadata and manifest caching |
| **Compaction Strategy** | Auto-optimize and bin-packing | Bin-packing with sort orders |
| **Query Pushdown** | Partition and statistics pruning | Partition and statistics pruning |

## G. Reading UC managed Tables as Iceberg: UniForm

UniForm bridges Delta and Iceberg by maintaining Iceberg-compatible metadata alongside the Delta transaction log, so external Iceberg clients can read the same data files without copying or converting them.
<br/><br/>

<div style="display: flex; align-items: center; justify-content: center; gap: 0; padding: 40px 24px 24px 24px; max-width: 900px; margin: 16px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; flex-wrap: wrap">

  <div style="background: #00838f; color: #fff; border: 2px solid #006064; border-radius: 6px; padding: 14px 22px; font-weight: 600; font-size: 1em; white-space: nowrap">Iceberg Client</div>

  <div style="display: flex; align-items: center; min-width: 160px; padding: 0 4px; position: relative">
    <span style="position: absolute; bottom: 100%; left: 0; right: 9px; text-align: center; font-size: 1em; color: #555; font-style: italic; margin-bottom: 6px">reads as Iceberg</span>
    <div style="flex: 1; height: 2px; background: #555"></div>
    <div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div>
  </div>

  <div style="background: #fff3e0; color: #333; border: 2px solid #ff9800; border-radius: 999px; padding: 14px 28px; font-weight: 600; font-size: 1em; white-space: nowrap">UniForm</div>

  <div style="display: flex; align-items: center; padding: 0 4px">
    <div style="width: 60px; height: 2px; background: #555"></div>
    <div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div>
  </div>

  <div style="background: #0277bd; color: #fff; border: 2px solid #01579b; border-radius: 6px; padding: 14px 22px; font-weight: 600; font-size: 1em; white-space: nowrap">UC managed Table</div>
</div>

### Universal Format - Cross-Format Data Access

UniForm enables external Iceberg clients to read UC managed tables **without data conversion or duplication** - just metadata translation.

| Feature | Detail |
|---------|--------|
| **No Duplication** | Only metadata is translated, not data files |
| **Transparent** | Databricks SQL and Spark query both formats without conversion |
| **Governance Preserved** | Existing security and governance policies apply |
| **Runtime Support** | Databricks Runtime 14.3 and above |
| **Migration Path** | Enables gradual migration between formats |

<div style="font-size: 1em; border-left: 4px solid #009688; background: #e0f2f1; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #00695c; font-size: 1.1em;">UniForm in Practice</strong>
            <p style="margin: 8px 0 0 0; color: #333;">UniForm is the recommended approach when you want to maintain Delta as your primary format while enabling external Iceberg clients to read the same data.</p>
        </div>
    </div>
</div>

## H. Evolution of the Iceberg Table Format

Iceberg has continued to evolve since v2 introduced row-level deletes, and v3 (now in Public Preview on Databricks Runtime 17.3 and above) is the version most new tables should target.

<div style="display: flex; align-items: stretch; justify-content: center; gap: 8px; padding: 24px 16px; max-width: 1000px; margin: 16px auto; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif; font-size: 1em; overflow-x: auto">

  <div style="flex: 0 0 180px; background: #fff3e0; border: 2px solid #ff9800; border-radius: 6px; padding: 16px; display: flex; flex-direction: column; justify-content: center; text-align: center">
    <div style="font-weight: 600; font-size: 1em; color: #333; margin-bottom: 6px">Iceberg v1</div>
    <div style="font-size: 1em; color: #555; line-height: 1.5">Foundation</div>
  </div>

  <div style="display: flex; align-items: center; align-self: center; flex: 0 0 auto">
    <div style="width: 18px; height: 2px; background: #555"></div>
    <div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div>
  </div>

  <div style="flex: 0 0 180px; background: #e3f2fd; border: 2px solid #1976d2; border-radius: 6px; padding: 16px; display: flex; flex-direction: column; justify-content: center; text-align: center">
    <div style="font-weight: 600; font-size: 1em; color: #333; margin-bottom: 6px">Iceberg v2</div>
    <div style="font-size: 1em; color: #555; line-height: 1.5">Row-level deletes<br/>Positional deletes</div>
  </div>

  <div style="display: flex; align-items: center; align-self: center; flex: 0 0 auto">
    <div style="width: 18px; height: 2px; background: #555"></div>
    <div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div>
  </div>

  <div style="flex: 0 0 180px; background: #e8f5e9; border: 2px solid #4caf50; border-radius: 6px; padding: 16px; display: flex; flex-direction: column; justify-content: center; text-align: center">
    <div style="font-weight: 600; font-size: 1em; color: #333; margin-bottom: 6px">Iceberg v3</div>
    <div style="font-size: 1em; color: #555; line-height: 1.5">Deletion vectors<br/><code>VARIANT</code> type<br/>Row lineage</div>
  </div>

  <div style="display: flex; align-items: center; align-self: center; flex: 0 0 auto">
    <div style="width: 18px; height: 2px; background: #555"></div>
    <div style="width: 0; height: 0; border-top: 6px solid transparent; border-bottom: 6px solid transparent; border-left: 9px solid #555"></div>
  </div>

  <div style="flex: 0 0 180px; background: #f3e5f5; border: 2px dashed #9c27b0; border-radius: 6px; padding: 16px; display: flex; flex-direction: column; justify-content: center; text-align: center">
    <div style="font-weight: 600; font-size: 1em; color: #333; margin-bottom: 6px">Iceberg v4</div>
    <div style="font-size: 1em; color: #555; line-height: 1.5; font-style: italic">(coming soon)</div>
  </div>

</div>

**v3 highlights worth knowing now:**
- **Deletion Vectors** - row-level deletes without rewriting data files (also available in Delta)
- **`VARIANT` data type** - native semi-structured/JSON storage and querying
- **Row Lineage** - automatic, required tracking of incremental row-level changes

Most users today will simply create their tables at format-version 3 and benefit from these improvements without thinking about them. Iceberg v4 sits beyond the scope of this course but appears on the timeline above so the trajectory is clear.

## Key Takeaways

- **Delta and Iceberg** both provide ACID transactions, schema evolution, and time travel on top of columnar storage
- **Delta** uses a JSON transaction log; **Iceberg** uses a layered metadata and manifest architecture
- Both formats support equivalent optimization techniques (clustering, statistics, data skipping, cleanup)
- **UniForm** lets UC managed tables be read by Iceberg clients without data duplication

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>